# v5 Candidate Scorer Training

This notebook trains and evaluates the lightweight candidate scorer. It reads `CANDIDATES_CSV` if set, otherwise `notebooks/v5/data/candidates_v5.csv`, otherwise the newest root `data/<run_start_timestamp>/candidates_v5.csv`. Outputs go to `notebooks/v5/exports/` and are uploaded to Hugging Face under `v5/`.

In [ ]:
import os
from getpass import getpass

HF_REPO_ID = "devaanshpa/orbit-wars-agent"
HF_REPO_TYPE = "model"
HF_REMOTE_PREFIX = "v5"

HF_TOKEN = getpass("HF_TOKEN: ").strip()
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required to upload v5 artifacts")
os.environ["HF_TOKEN"] = HF_TOKEN

try:
    from huggingface_hub import HfApi
except ModuleNotFoundError:
    %pip install -q huggingface_hub
    from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)
print(f"Authenticated. Artifacts will upload to {HF_REPO_ID}/{HF_REMOTE_PREFIX}/")

In [ ]:
from pathlib import Path
import csv
import json
import math
import os
import random
import time

BASE_DIR = Path("notebooks/v5") if (Path.cwd() / "notebooks" / "v5").exists() else Path.cwd()
env_data_path = os.environ.get("CANDIDATES_CSV", "").strip()
DATA_PATH = Path(env_data_path) if env_data_path else BASE_DIR / "data" / "candidates_v5.csv"
if not DATA_PATH.exists():
    search_roots = [Path("data"), Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd().parent.parent / "data"]
    candidates = []
    seen = set()
    for root_data in search_roots:
        if not root_data.exists():
            continue
        for candidate in root_data.glob("*/candidates_v5.csv"):
            resolved = candidate.resolve()
            if resolved not in seen:
                seen.add(resolved)
                candidates.append(candidate)
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        DATA_PATH = candidates[0]
    else:
        print("No local candidates_v5.csv found; downloading newest dataset from Hugging Face...")
        try:
            from huggingface_hub import hf_hub_download
        except ModuleNotFoundError:
            %pip install -q huggingface_hub
            from huggingface_hub import hf_hub_download
        repo_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
        remote_csvs = sorted(
            repo_path for repo_path in repo_files
            if repo_path.startswith("data/") and repo_path.endswith("/candidates_v5.csv")
        )
        if not remote_csvs:
            raise FileNotFoundError("No candidates_v5.csv found locally or in Hugging Face data/<run_start_timestamp>/ folders.")
        remote_csv = remote_csvs[-1]
        downloaded = hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            filename=remote_csv,
            token=HF_TOKEN,
        )
        DATA_PATH = Path(downloaded)
        print(f"Downloaded {remote_csv}")

EXPORT_DIR = BASE_DIR / "exports"
GRAPH_DIR = EXPORT_DIR / "graphs"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using training data: {DATA_PATH}")

with DATA_PATH.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
if not rows:
    raise RuntimeError("Training CSV has no rows")

metadata_cols = {"label", "game_id", "candidate_id", "version"}
feature_names = [name for name in rows[0] if name not in metadata_cols]
x_raw = [[float(row.get(name, 0.0) or 0.0) for name in feature_names] for row in rows]
y = [max(0.0, min(1.0, float(row["label"]))) for row in rows]

indices = list(range(len(rows)))
random.Random(42).shuffle(indices)
split = max(1, int(len(indices) * 0.8)) if len(indices) >= 5 else len(indices)
train_indices = indices[:split]
valid_indices = indices[split:] or indices[:1]
train_raw = [x_raw[i] for i in train_indices]
valid_raw = [x_raw[i] for i in valid_indices]
train_y = [y[i] for i in train_indices]
valid_y = [y[i] for i in valid_indices]

means = [sum(col) / len(col) for col in zip(*train_raw)]
scales = []
for j, mean in enumerate(means):
    var = sum((row[j] - mean) ** 2 for row in train_raw) / max(1, len(train_raw) - 1)
    scales.append(max(1e-6, math.sqrt(var)))

def normalize(items):
    return [[(row[j] - means[j]) / scales[j] for j in range(len(feature_names))] for row in items]

train_x = normalize(train_raw)
valid_x = normalize(valid_raw)
all_x = normalize(x_raw)

pos_count = sum(1 for label in train_y if label >= 0.5)
neg_count = len(train_y) - pos_count
pos_weight = len(train_y) / (2.0 * pos_count) if pos_count else 0.0
neg_weight = len(train_y) / (2.0 * neg_count) if neg_count else 0.0

weights = [0.0] * len(feature_names)
bias = 0.0
learning_rate = 0.04
l2 = 0.001
epochs = 350
history = []
train_started_at = time.time()
print(f"Loaded {len(rows)} rows with {len(feature_names)} features")
print(f"train_rows={len(train_x)} validation_rows={len(valid_x)} positive_rate={sum(1 for label in y if label >= 0.5) / len(y):.3f}")
print(f"class_weights positive={pos_weight:.3f} negative={neg_weight:.3f}")

def sigmoid(z):
    if z >= 0:
        ez = math.exp(-min(50.0, z))
        return 1.0 / (1.0 + ez)
    ez = math.exp(max(-50.0, z))
    return ez / (1.0 + ez)

def predict_batch(x_rows):
    return [sigmoid(bias + sum(w * v for w, v in zip(weights, features))) for features in x_rows]

def log_loss(preds, labels):
    return -sum(label * math.log(max(1e-9, p)) + (1.0 - label) * math.log(max(1e-9, 1.0 - p)) for p, label in zip(preds, labels)) / len(labels)

def accuracy(preds, labels):
    return sum((p >= 0.5) == (label >= 0.5) for p, label in zip(preds, labels)) / len(labels)

for epoch in range(epochs):
    grad_w = [0.0] * len(weights)
    grad_b = 0.0
    for features, label in zip(train_x, train_y):
        pred = sigmoid(bias + sum(w * v for w, v in zip(weights, features)))
        sample_weight = pos_weight if label >= 0.5 else neg_weight
        err = (pred - label) * sample_weight
        grad_b += err
        for j, value in enumerate(features):
            grad_w[j] += err * value
    n = len(train_x)
    bias -= learning_rate * grad_b / n
    for j in range(len(weights)):
        weights[j] -= learning_rate * ((grad_w[j] / n) + l2 * weights[j])
    if epoch == 0 or (epoch + 1) % 10 == 0 or epoch == epochs - 1:
        train_preds_epoch = predict_batch(train_x)
        valid_preds_epoch = predict_batch(valid_x)
        train_loss = log_loss(train_preds_epoch, train_y)
        valid_loss = log_loss(valid_preds_epoch, valid_y)
        train_acc = accuracy(train_preds_epoch, train_y)
        valid_acc = accuracy(valid_preds_epoch, valid_y)
        elapsed = time.time() - train_started_at
        history.append({
            "epoch": epoch + 1,
            "train_logloss": train_loss,
            "valid_logloss": valid_loss,
            "train_accuracy": train_acc,
            "valid_accuracy": valid_acc,
            "elapsed_seconds": elapsed,
        })
        print(
            f"epoch={epoch + 1:03d}/{epochs} "
            f"train_loss={train_loss:.4f} valid_loss={valid_loss:.4f} "
            f"train_acc={train_acc:.3f} valid_acc={valid_acc:.3f} "
            f"elapsed={elapsed:.1f}s",
            flush=True,
        )

train_preds = predict_batch(train_x)
valid_preds = predict_batch(valid_x)
all_preds = predict_batch(all_x)
metrics = {
    "rows": len(rows),
    "train_rows": len(train_x),
    "validation_rows": len(valid_x),
    "positive_rate": sum(1 for label in y if label >= 0.5) / len(y),
    "train_positive_rate": pos_count / len(train_y),
    "validation_positive_rate": sum(1 for label in valid_y if label >= 0.5) / len(valid_y),
    "positive_class_weight": pos_weight,
    "negative_class_weight": neg_weight,
    "train_accuracy": accuracy(train_preds, train_y),
    "validation_accuracy": accuracy(valid_preds, valid_y),
    "train_logloss": log_loss(train_preds, train_y),
    "validation_logloss": log_loss(valid_preds, valid_y),
}
artifact = {
    "version": "v5",
    "created_at": int(time.time()),
    "source_csv": str(DATA_PATH),
    "model_type": "logistic_linear_candidate_scorer",
    "features": feature_names,
    "mean": dict(zip(feature_names, means)),
    "scale": dict(zip(feature_names, scales)),
    "bias": bias,
    "weights": dict(zip(feature_names, weights)),
    "metrics": metrics,
}
(EXPORT_DIR / "model_weights_v5.json").write_text(json.dumps(artifact, indent=2, sort_keys=True), encoding="utf-8")
(EXPORT_DIR / "feature_schema_v5.json").write_text(json.dumps({"features": feature_names}, indent=2), encoding="utf-8")
(EXPORT_DIR / "metrics_v5.json").write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8")
(EXPORT_DIR / "training_history_v5.json").write_text(json.dumps(history, indent=2, sort_keys=True), encoding="utf-8")
with (EXPORT_DIR / "predictions_v5.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["row_index", "label", "prediction", "split"])
    valid_set = set(valid_indices)
    for i, pred in enumerate(all_preds):
        writer.writerow([i, y[i], pred, "validation" if i in valid_set else "train"])
print(json.dumps(metrics, indent=2, sort_keys=True))
print("Top absolute weights:")
for name, value in sorted(zip(feature_names, weights), key=lambda item: abs(item[1]), reverse=True)[:20]:
    print(f"{name:32s} {value: .5f}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    %pip install -q matplotlib
    import matplotlib.pyplot as plt

plot_paths = []
epochs_seen = [item["epoch"] for item in history]
plt.figure(figsize=(9, 5))
plt.plot(epochs_seen, [item["train_logloss"] for item in history], label="train")
plt.plot(epochs_seen, [item["valid_logloss"] for item in history], label="validation")
plt.xlabel("epoch")
plt.ylabel("log loss")
plt.title("v5 candidate scorer loss")
plt.legend()
plt.grid(alpha=0.25)
path = GRAPH_DIR / "loss_curve_v5.png"
plt.tight_layout()
plt.savefig(path, dpi=160)
plot_paths.append(path)
plt.show()

plt.figure(figsize=(9, 5))
plt.hist([p for p, label in zip(all_preds, y) if label >= 0.5], bins=25, alpha=0.65, label="positive")
plt.hist([p for p, label in zip(all_preds, y) if label < 0.5], bins=25, alpha=0.65, label="negative")
plt.xlabel("predicted probability")
plt.ylabel("rows")
plt.title("v5 prediction distribution")
plt.legend()
path = GRAPH_DIR / "prediction_histogram_v5.png"
plt.tight_layout()
plt.savefig(path, dpi=160)
plot_paths.append(path)
plt.show()

top_weights = sorted(zip(feature_names, weights), key=lambda item: abs(item[1]), reverse=True)[:25]
names = [name for name, _ in reversed(top_weights)]
values = [value for _, value in reversed(top_weights)]
plt.figure(figsize=(10, max(5, len(names) * 0.28)))
plt.barh(names, values)
plt.xlabel("weight")
plt.title("v5 top feature weights")
plt.axvline(0, color="black", linewidth=0.8)
path = GRAPH_DIR / "feature_weights_v5.png"
plt.tight_layout()
plt.savefig(path, dpi=160)
plot_paths.append(path)
plt.show()

print("Saved graphs:")
for path in plot_paths:
    print(path)

In [ ]:
api.upload_folder(
    folder_path=str(EXPORT_DIR),
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    path_in_repo=HF_REMOTE_PREFIX,
    commit_message="Upload v5 Orbit Wars training artifacts and graphs",
)
print(f"Uploaded {EXPORT_DIR} to https://huggingface.co/{HF_REPO_ID}/tree/main/{HF_REMOTE_PREFIX}")